using the maximum Q-value estimate during bootstrapping could be problematic. as the bootstraping for max stimate is so noisy


different method for alg:
- instead of directly go with max q value in target, we just find max based on main_q and then calculate target
 - chosen_action = model_main(next_state).argmax(1)
 - chosen_value = model_target(next_state).gather(chosen_action)


- Clipped Double Q-learning.


In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.autograd as autograd

import numpy as np
import gymnasium as gym
import random
from collections import deque

In [8]:
max_buff_len = 10000

memory = deque(maxlen=max_buff_len)

env_id = "CartPole-v0"
MAX_EPISODES = 1000
MAX_STEPS = 500
BATCH_SIZE = 32

env = gym.make(env_id)
gamma = 0.99
epsilon_decay = 0.995

output_dim = env.action_space.n

In [3]:
state,info = env.reset()

In [4]:
state

array([ 0.01534546, -0.0357904 ,  0.0320452 , -0.04175528], dtype=float32)

In [6]:
class Q_Network(nn.Module):
    
    def __init__(self,input_dim,output_n):
        super(Q_Network,self).__init__()

        self.fc1 = nn.Linear(input_dim,128)
        self.fc2 = nn.Linear(128,128)
        self.fc3 = nn.Linear(128,output_n)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        return self.fc3(x)

    

In [11]:
# update model
# calculate the lost and ..

class Agent:
    
    def __init__(self,env):

        self.input_dim = env.observation_space.shape[0]
        self.output_dim = env.action_space.n
        
        self.model_online = Q_Network(self.input_dim,self.output_dim)

        self.model_target = Q_Network(self.input_dim,self.output_dim)

        self.model_target.load_state_dict(self.model_online.set_extra_state)

        optimizer = torch.optim.Adam(self.model_online.parameters(),lr = 1e-3)


    def optimize(self):
        if len(memory) < batch_size:
            return
    
        batch = random.sample(memory,batch_size)
        state_batch, action_batch, reward_batch, next_state_batch, done_batch = zip(*batch)
    
    
        state_batch = torch.FloatTensor(state_batch)
        action_batch = torch.LongTensor(action_batch).unsqueeze(1)
        reward_batch = torch.FloatTensor(reward_batch)
        next_state_batch = torch.FloatTensor(next_state_batch)
        done_batch = torch.FloatTensor(done_batch)

        #this is the place that double Q happen
        Q_values = self.model_online(state_batch)
        chosen_Q = Q_values.gather(1,action_batch).squeeze()

        with torch.no_grad():
            
            Q_actions = self.model_online(next_state_batch).argmax(1,keepdim=True)
            Q_next_state = self.model_target(next_state_batch).gather(1,Q_actions)
            Q_target = reward_batch + gamma * Q_next_state * (1-done)


        
        loss = nn.MSELoss(chosen_Q,Q_target)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()


    

        
        